# Concierge Intent Classifier — Public Dataset Training and Artifact Export

This notebook trains the Concierge classifier used by the routing layer. The classifier predicts whether an inbound visitor message should be treated as spam, a question, a lead, or an escalation request. The dataset is built from public labeled text-classification sources, then evaluated with a classical ML model, a small DL model exported to ONNX, and an optional hosted-API LLM zero-shot baseline. The final section exports artifacts, evaluation results, metadata, and a model card for the lean model-server.

## 1. Install notebook-only dependencies

This notebook runs in Colab, so training dependencies such as PyTorch are allowed here. These dependencies must not be copied into the production model-server container. The production model-server should use only lean runtime dependencies such as `onnxruntime`, `scikit-learn`, `joblib`, and `numpy`.

In [ ]:
!pip -q install datasets scikit-learn pandas numpy joblib onnx onnxruntime onnxscript openai pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.9 MB/s eta 0:00:00


## 2. Import libraries and define experiment configuration

This cell defines the fixed label set, output folders, random seed, confidence threshold, and experiment metadata. The four labels map directly to the Concierge router actions.

In [ ]:
import hashlib
import json
import os
import random
import re
import shutil
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import yaml
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

import onnxruntime as ort

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LABELS = ["spam", "question", "lead", "escalate"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

PROJECT_NAME = "concierge-intent-classifier"
MODEL_VERSION = "intent-router-v1"
CONFIDENCE_THRESHOLD = 0.70

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACT_DIR = ROOT_DIR / "artifacts"
REPORT_DIR = ROOT_DIR / "reports"
EXPORT_DIR = ROOT_DIR / "export"

for directory in [PROCESSED_DIR, ARTIFACT_DIR, REPORT_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT_NAME}")
print(f"Labels: {LABELS}")
print(f"Output root: {ROOT_DIR}")

Project: concierge-intent-classifier
Labels: ['spam', 'question', 'lead', 'escalate']
Output root: /content/concierge_classifier


## 3. Define reusable utility functions

The notebook uses helper functions for text cleanup, hashing, JSON export, and latency measurement. Hashing is important because the exported model artifact hash is pinned in the model card.

In [ ]:
def normalize_text(text: str) -> str:
    """Normalize whitespace and remove control characters."""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\x00", "")
    return text.strip()


def sha256_file(path: Path) -> str:
    """Compute the SHA-256 hash of a file."""
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_directory_files(paths: list[Path]) -> str:
    """Compute a combined SHA-256 hash for a list of files."""
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.name):
        digest.update(path.name.encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a JSON file with stable formatting."""
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def measure_latency_ms(predict_fn, texts: list[str], runs: int = 3) -> dict[str, float]:
    """Measure average and p95 prediction latency in milliseconds per message."""
    durations: list[float] = []

    for _ in range(runs):
        for text in texts:
            start = time.perf_counter()
            predict_fn([text])
            durations.append((time.perf_counter() - start) * 1000)

    return {
        "avg_ms": float(np.mean(durations)),
        "p95_ms": float(np.percentile(durations, 95)),
    }


def print_section(title: str) -> None:
    """Print a readable notebook section heading."""
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

## 4. Load the public SMS Spam Collection dataset

The spam class is built from a public labeled SMS spam dataset. Spam messages should be dropped before storage or expensive agent execution.

In [ ]:
SPAM_DATA_URL = (
    "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/"
    "master/data/sms.tsv"
)

sms_df = pd.read_csv(
    SPAM_DATA_URL,
    sep="\t",
    header=None,
    names=["source_label", "text"],
)

sms_df["text"] = sms_df["text"].map(normalize_text)

spam_df = (
    sms_df[sms_df["source_label"] == "spam"][["text"]]
    .drop_duplicates()
    .assign(
        label="spam",
        source_dataset="SMS Spam Collection",
        source_label="spam",
    )
    .reset_index(drop=True)
)

print(spam_df.head())
print(f"Spam examples loaded: {len(spam_df)}")

                                                text label  \
0  Free entry in 2 a wkly comp to win FA Cup fina...  spam   
1  FreeMsg Hey there darling it's been 3 week's n...  spam   
2  WINNER!! As a valued network customer you have...  spam   
3  Had your mobile 11 months or more? U R entitle...  spam   
4  SIX chances to win CASH! From 100 to 20,000 po...  spam   

        source_dataset source_label  
0  SMS Spam Collection         spam  
1  SMS Spam Collection         spam  
2  SMS Spam Collection         spam  
3  SMS Spam Collection         spam  
4  SMS Spam Collection         spam  
Spam examples loaded: 642


## 5. Load the public CLINC150 intent dataset

The non-spam routing labels are created from a public intent-classification dataset. CLINC150 contains many user intent labels, which are collapsed into the Concierge routing labels `question`, `lead`, and `escalate`.

In [ ]:
# Public CLINC150-style intent dataset from Hugging Face.

clinc_dataset = load_dataset("DeepPavlov/clinc_oos", "plus")


def find_text_column(frame: pd.DataFrame) -> str:
    """Find the text column used by the public dataset."""
    for candidate in ["text", "utterance", "query", "sentence"]:
        if candidate in frame.columns:
            return candidate

    raise ValueError(f"Could not find a text column. Columns: {frame.columns.tolist()}")


def find_label_column(frame: pd.DataFrame) -> str:
    """Find the intent label column used by the public dataset."""
    for candidate in ["intent", "label", "category"]:
        if candidate in frame.columns:
            return candidate

    raise ValueError(f"Could not find a label column. Columns: {frame.columns.tolist()}")


def clinc_split_to_frame(split_name: str) -> pd.DataFrame:
    """Convert one CLINC split into a dataframe with readable intent labels."""
    split = clinc_dataset[split_name]
    frame = pd.DataFrame(split)

    text_column = find_text_column(frame)
    label_column = find_label_column(frame)

    frame["text"] = frame[text_column].map(normalize_text)

    label_feature = split.features[label_column]
    if hasattr(label_feature, "names") and label_feature.names is not None:
        frame["source_label"] = frame[label_column].map(lambda idx: label_feature.names[idx])
    else:
        frame["source_label"] = frame[label_column].astype(str)

    frame["source_dataset"] = "DeepPavlov/clinc_oos"

    return frame[["text", "source_label", "source_dataset"]]


clinc_frames = [
    clinc_split_to_frame(split_name)
    for split_name in clinc_dataset.keys()
]

clinc_df = (
    pd.concat(clinc_frames, ignore_index=True)
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print(clinc_df.head())
print(f"Available splits: {list(clinc_dataset.keys())}")
print(f"CLINC examples loaded: {len(clinc_df)}")
print(f"Number of public CLINC intent labels: {clinc_df['source_label'].nunique()}")

print("\nSample intent labels:")
print(sorted(clinc_df["source_label"].unique())[:50])

plus/train-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

plus/validation-00000-of-00001.parquet:   0%|          | 0.00/74.5k [00:00<?, ?B/s]

plus/test-00000-of-00001.parquet:   0%|          | 0.00/133k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

                                                text source_label  \
0  what expression would i use to say i love you ...           61   
1  can you tell me how to say 'i do not speak muc...           61   
2  what is the equivalent of, 'life is good' in f...           61   
3  tell me how to say, 'it is a beautiful morning...           61   
4  if i were mongolian, how would i say that i am...           61   

         source_dataset  
0  DeepPavlov/clinc_oos  
1  DeepPavlov/clinc_oos  
2  DeepPavlov/clinc_oos  
3  DeepPavlov/clinc_oos  
4  DeepPavlov/clinc_oos  
Available splits: ['train', 'validation', 'test']
CLINC examples loaded: 23845
Number of public CLINC intent labels: 151

Sample intent labels:
['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135'

## 6. Map public intent labels to Concierge router labels

The public CLINC150 labels are more detailed than the product router needs. This cell transparently maps groups of public intent labels into `question`, `lead`, and `escalate`. The mapping is intentionally printed and saved so it can be reviewed.

In [ ]:
# Stricter text-based mapping from public CLINC examples into Concierge router labels.
# The goal is to avoid broad keywords like "hotel", "lost", or "help me" causing false labels.

TRANSLATION_OR_LANGUAGE_PATTERNS = [
    "how do i say",
    "how would i say",
    "tell me how",
    "what words would i use",
    "translate",
    "in finnish",
    "in japanese",
    "in russian",
    "in spanish",
    "in french",
    "in german",
    "meaning of",
]

ESCALATE_PATTERNS = [
    "talk to a human",
    "speak to a human",
    "speak with a human",
    "talk to a person",
    "speak to a person",
    "real person",
    "human agent",
    "support agent",
    "customer support",
    "representative",
    "manager",
    "complaint",
    "urgent support",
    "emergency",
    "fraud",
    "stolen card",
    "lost card",
    "lost luggage",
    "freeze my card",
    "freeze my account",
    "card was declined",
    "card declined",
    "not working",
    "file a claim",
    "open a ticket",
    "support ticket",
]

LEAD_PATTERNS = [
    "book a demo",
    "schedule a demo",
    "request a demo",
    "get a quote",
    "request a quote",
    "contact sales",
    "talk to sales",
    "pricing",
    "enterprise plan",
    "buy a plan",
    "purchase a plan",
    "schedule a meeting",
    "book a meeting",
    "schedule an appointment",
    "book an appointment",
    "make a reservation",
    "book a reservation",
    "reserve a table",
    "book a table",
    "book a flight",
    "book a hotel",
    "book a room",
    "rent a car",
    "car rental",
    "schedule a vacation",
    "book a vacation",
    "schedule vacation",
    "request vacation",
    "schedule pto",
    "pto request",
    "insurance plan",
    "insurance policy",
    "new renters' insurance",
    "new auto insurance",
]

QUESTION_STARTERS = (
    "what ",
    "how ",
    "when ",
    "where ",
    "why ",
    "who ",
    "can i ",
    "could i ",
    "do i ",
    "does ",
    "is ",
    "are ",
    "tell me ",
    "show me ",
    "find ",
    "search ",
)


def contains_any(text: str, patterns: list[str]) -> bool:
    """Return True if any phrase pattern appears in the text."""
    return any(pattern in text for pattern in patterns)


def map_text_to_router_label(text: str) -> str | None:
    """Map public dataset text into a Concierge router label."""
    normalized_text = normalize_text(text).lower()

    # Translation/language examples should not become lead or escalate just
    # because they contain words like hotel, lost, or notebook.
    if contains_any(normalized_text, TRANSLATION_OR_LANGUAGE_PATTERNS):
        return "question"

    # Escalation must be explicit: human handoff, serious support issue,
    # complaint, fraud, stolen/lost card/luggage, ticket, or urgent support.
    if contains_any(normalized_text, ESCALATE_PATTERNS):
        return "escalate"

    # Lead means booking, sales, quote, demo, appointment, pricing, or purchase intent.
    if contains_any(normalized_text, LEAD_PATTERNS):
        return "lead"

    # General informational requests are questions.
    if "?" in normalized_text or normalized_text.startswith(QUESTION_STARTERS):
        return "question"

    return None


clinc_mapped_df = clinc_df.copy()
clinc_mapped_df["label"] = clinc_mapped_df["text"].map(map_text_to_router_label)

clinc_mapped_df = (
    clinc_mapped_df.dropna(subset=["label"])
    [["text", "label", "source_dataset", "source_label"]]
    .reset_index(drop=True)
)

print_section("Mapped public CLINC examples")
display(
    clinc_mapped_df["label"]
    .value_counts()
    .reindex(LABELS, fill_value=0)
    .to_frame("count")
)

print_section("Sample examples by mapped label")
for label in ["question", "lead", "escalate"]:
    print(f"\n[{label}]")
    samples = clinc_mapped_df[clinc_mapped_df["label"] == label].sample(
        n=min(10, len(clinc_mapped_df[clinc_mapped_df["label"] == label])),
        random_state=SEED,
    )
    for _, row in samples.iterrows():
        print(f"- {row['text']}")


Mapped public CLINC examples


,count
label,
spam,0
question,9795
lead,455
escalate,264



Sample examples by mapped label

[question]
- what is the process to move money from one account to another
- what phrase means goodbye in hawaii
- what do i do to get cach back for points on my discover card
- how long does it take to get your yellow belt in karate
- what exactly can you help me with
- how might i go about jump starting a car
- how much does it cost to use the parking garage downtown
- how long do i cook this for
- what kind of person are you, a cat or dog
- how are things

[lead]
- are there any places nearby i can rent a car at
- i really need to switch to a new insurance plan
- i need to change my insurance policy, do you know how
- reserve a table for 3 at 7 for olive garden
- would you please make a reservation for 2 at olive garden for 5 pm today
- i want to book a flight from hawaii to new york on july 8th and returning on july 10th
- find and book a hotel in md, baltimore starting on the 7th to the 9th
- could you submit a pto request for me from dates mar 3 

In [ ]:
print_section("Combined public dataset label counts")

combined_preview_df = pd.concat([spam_df, clinc_mapped_df], ignore_index=True)

display(
    combined_preview_df["label"]
    .value_counts()
    .reindex(LABELS, fill_value=0)
    .to_frame("count")
)


Combined public dataset label counts


,count
label,
spam,642
question,9795
lead,455
escalate,264


## 7. Validate public dataset coverage

This cell checks that each router class has enough examples from public datasets. If a class is too small, the notebook stops instead of silently replacing the public dataset with synthetic data.

In [ ]:
public_df = pd.concat(
    [spam_df, clinc_mapped_df],
    ignore_index=True,
)

public_df["text"] = public_df["text"].map(normalize_text)
public_df = (
    public_df.dropna(subset=["text", "label"])
    .drop_duplicates(subset=["text"])
    .query("label in @LABELS")
    .reset_index(drop=True)
)

class_counts = public_df["label"].value_counts().reindex(LABELS).fillna(0).astype(int)
display(class_counts.to_frame("public_examples"))

minimum_required_per_class = 100

missing_or_small = [
    label
    for label, count in class_counts.items()
    if count < minimum_required_per_class
]

if missing_or_small:
    raise ValueError(
        "Not enough public examples for these labels: "
        f"{missing_or_small}. Review the mapping keywords or choose another "
        "public intent dataset before training."
    )

print("Public dataset coverage is sufficient for all labels.")

,public_examples
label,
spam,642
question,9795
lead,455
escalate,264


Public dataset coverage is sufficient for all labels.


## 8. Balance and split the dataset

The dataset is balanced by class, then split into train, validation, and test sets using stratification. The held-out test set is not used during model training or model selection.

In [ ]:
target_per_class = min(600, public_df["label"].value_counts().min())

balanced_df = (
    public_df.groupby("label", group_keys=False)
    .apply(lambda group: group.sample(n=target_per_class, random_state=SEED))
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)

train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.30,
    random_state=SEED,
    stratify=balanced_df["label"],
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

train_path = PROCESSED_DIR / "classifier_train.csv"
valid_path = PROCESSED_DIR / "classifier_valid.csv"
test_path = PROCESSED_DIR / "classifier_test.csv"
full_path = PROCESSED_DIR / "classifier_full.csv"

train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)
balanced_df.to_csv(full_path, index=False)

dataset_hash = sha256_directory_files([train_path, valid_path, test_path])

print_section("Dataset sizes")
print(f"Train: {len(train_df)}")
print(f"Valid: {len(valid_df)}")
print(f"Test:  {len(test_df)}")

print_section("Class distribution")
display(
    pd.DataFrame(
        {
            "train": train_df["label"].value_counts(),
            "valid": valid_df["label"].value_counts(),
            "test": test_df["label"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
    .reindex(LABELS)
)

print(f"\nDataset SHA-256: {dataset_hash}")


Dataset sizes
Train: 739
Valid: 158
Test:  159

Class distribution


/tmp/ipykernel_3471/2981870486.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=target_per_class, random_state=SEED))


,train,valid,test
label,,,
spam,185,39,40
question,184,40,40
lead,185,39,40
escalate,185,40,39



Dataset SHA-256: 894e9a9c6ac512789458f5385839347eb68ace52fc2b6158e89ceb53d54cf350


## 9. Inspect dataset examples

A short manual inspection helps verify that the public intent labels were collapsed into the correct router classes.

In [ ]:
print_section("Sample examples by router label")

for label in LABELS:
    print(f"\n[{label}]")
    samples = train_df[train_df["label"] == label].sample(
        n=min(8, len(train_df[train_df["label"] == label])),
        random_state=SEED,
    )

    for _, row in samples.iterrows():
        source = row.get("source_label", "")
        print(f"- {row['text']}  ({source})")


Sample examples by router label

[spam]
- 5p 4 alfie Moon's Children in need song on ur mob. Tell ur m8s. Txt Tone charity to 8007 for Nokias or Poly charity for polys: zed 08701417012 profit 2 charity.  (spam)
- PRIVATE! Your 2003 Account Statement for shows 800 un-redeemed S. I. M. points. Call 08715203694 Identifier Code: 40533 Expires 31/10/04  (spam)
- INTERFLORA - It's not too late to order Interflora flowers for christmas call 0800 505060 to place your order before Midnight tomorrow.  (spam)
- You are being contacted by our dating service by someone you know! To find out who it is, call from a land line 09050000878. PoBox45W2TG150P  (spam)
- Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort time 09099726429 JANINExx Calls£1/minMobsmoreLKPOBOX177HP51FL  (spam)
- FREE RING TONE just text "POLYS" to 87131. Then every week get a new tone. 0870737910216yrs only £1.50/wk.  (spam)
- T-Mobile customer you may now claim your FREE CAMERA P

## 10. Define shared evaluation helpers

All models are evaluated with the same metrics. Macro-F1 is the main score because it gives equal weight to every class, including classes that may be harder or less frequent.

In [ ]:
@dataclass
class ModelResult:
    name: str
    macro_f1: float
    weighted_f1: float
    per_class_f1: dict[str, float]
    avg_latency_ms: float
    p95_latency_ms: float
    estimated_cost_usd: float | None
    notes: str


def evaluate_predictions(
    name: str,
    y_true: list[str],
    y_pred: list[str],
    latency: dict[str, float],
    estimated_cost_usd: float | None,
    notes: str,
) -> ModelResult:
    """Compute shared metrics for a classifier."""
    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )

    per_class_f1 = {
        label: float(report[label]["f1-score"])
        for label in LABELS
    }

    return ModelResult(
        name=name,
        macro_f1=float(f1_score(y_true, y_pred, labels=LABELS, average="macro")),
        weighted_f1=float(f1_score(y_true, y_pred, labels=LABELS, average="weighted")),
        per_class_f1=per_class_f1,
        avg_latency_ms=float(latency["avg_ms"]),
        p95_latency_ms=float(latency["p95_ms"]),
        estimated_cost_usd=estimated_cost_usd,
        notes=notes,
    )


def display_result(result: ModelResult) -> None:
    """Display one model result in a readable format."""
    print_section(result.name)
    print(f"Macro-F1:       {result.macro_f1:.4f}")
    print(f"Weighted-F1:    {result.weighted_f1:.4f}")
    print(f"Avg latency ms: {result.avg_latency_ms:.2f}")
    print(f"P95 latency ms: {result.p95_latency_ms:.2f}")
    print(f"Cost USD:       {result.estimated_cost_usd}")
    print("Per-class F1:")

    for label, score in result.per_class_f1.items():
        print(f"  {label:10s} {score:.4f}")

    print(f"Notes: {result.notes}")

## 11. Train the classical ML baseline

The classical model uses TF-IDF features with logistic regression. This is a strong baseline for short text classification and can be served cheaply with `scikit-learn` and `joblib`.

In [ ]:
classical_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
                max_features=20_000,
                strip_accents="unicode",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1_000,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ]
)

x_train = train_df["text"].tolist()
y_train = train_df["label"].tolist()
x_valid = valid_df["text"].tolist()
y_valid = valid_df["label"].tolist()
x_test = test_df["text"].tolist()
y_test = test_df["label"].tolist()

start = time.perf_counter()
classical_pipeline.fit(x_train, y_train)
training_seconds = time.perf_counter() - start

classical_test_pred = classical_pipeline.predict(x_test).tolist()

classical_latency = measure_latency_ms(
    classical_pipeline.predict,
    x_test[: min(100, len(x_test))],
    runs=5,
)

classical_result = evaluate_predictions(
    name="classical_tfidf_logistic_regression",
    y_true=y_test,
    y_pred=classical_test_pred,
    latency=classical_latency,
    estimated_cost_usd=0.0,
    notes=f"Training completed in {training_seconds:.2f} seconds. Exported with joblib.",
)

classical_artifact_path = ARTIFACT_DIR / "classifier.joblib"
joblib.dump(classical_pipeline, classical_artifact_path)

display_result(classical_result)
print(f"\nSaved classical artifact: {classical_artifact_path}")
print(f"Classical artifact SHA-256: {sha256_file(classical_artifact_path)}")


classical_tfidf_logistic_regression
Macro-F1:       0.9432
Weighted-F1:    0.9433
Avg latency ms: 0.95
P95 latency ms: 1.26
Cost USD:       0.0
Per-class F1:
  spam       0.9639
  question   0.9136
  lead       0.9620
  escalate   0.9333
Notes: Training completed in 0.43 seconds. Exported with joblib.

Saved classical artifact: /content/concierge_classifier/artifacts/classifier.joblib
Classical artifact SHA-256: e4f16974584128d61a9a9a07f197d6de5624fc7901edd20ffcf656ececf3d2d3


## 12. Prepare features for the small DL model

The small deep-learning model uses TF-IDF vectors as input. This keeps the model simple enough while still producing a real neural model that can be exported to ONNX.

In [ ]:
dl_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    max_features=5_000,
    strip_accents="unicode",
)

x_train_vec = dl_vectorizer.fit_transform(x_train).astype(np.float32).toarray()
x_valid_vec = dl_vectorizer.transform(x_valid).astype(np.float32).toarray()
x_test_vec = dl_vectorizer.transform(x_test).astype(np.float32).toarray()

y_train_ids = np.array([LABEL_TO_ID[label] for label in y_train], dtype=np.int64)
y_valid_ids = np.array([LABEL_TO_ID[label] for label in y_valid], dtype=np.int64)
y_test_ids = np.array([LABEL_TO_ID[label] for label in y_test], dtype=np.int64)

print(f"DL input shape: {x_train_vec.shape}")
print(f"Number of labels: {len(LABELS)}")

DL input shape: (739, 5000)
Number of labels: 4


## 13. Train the small neural classifier

The neural model is intentionally small. The goal is to satisfy the ML/DL comparison requirement without creating a heavy serving stack. PyTorch is used only in the notebook training environment.

In [ ]:
class IntentMLP(nn.Module):
    """Small MLP classifier for TF-IDF intent vectors."""

    def __init__(self, input_dim: int, output_dim: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, output_dim),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

input_dim = x_train_vec.shape[1]
output_dim = len(LABELS)

model = IntentMLP(input_dim=input_dim, output_dim=output_dim).to(device)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(LABELS)),
    y=y_train_ids,
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_dataset = TensorDataset(
    torch.tensor(x_train_vec, dtype=torch.float32),
    torch.tensor(y_train_ids, dtype=torch.long),
)
valid_dataset = TensorDataset(
    torch.tensor(x_valid_vec, dtype=torch.float32),
    torch.tensor(y_valid_ids, dtype=torch.long),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

best_valid_macro_f1 = -1.0
best_state_dict = None
epochs = 20
patience = 4
epochs_without_improvement = 0

start = time.perf_counter()

for epoch in range(1, epochs + 1):
    model.train()
    train_losses = []

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_features)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    model.eval()
    valid_preds: list[int] = []

    with torch.no_grad():
        for batch_features, _ in valid_loader:
            batch_features = batch_features.to(device)
            logits = model(batch_features)
            preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()
            valid_preds.extend(preds)

    valid_pred_labels = [ID_TO_LABEL[pred] for pred in valid_preds]
    valid_macro_f1 = f1_score(
        y_valid,
        valid_pred_labels,
        labels=LABELS,
        average="macro",
    )

    print(
        f"Epoch {epoch:02d} | "
        f"loss={np.mean(train_losses):.4f} | "
        f"valid_macro_f1={valid_macro_f1:.4f}"
    )

    if valid_macro_f1 > best_valid_macro_f1:
        best_valid_macro_f1 = valid_macro_f1
        best_state_dict = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

training_seconds = time.perf_counter() - start

if best_state_dict is None:
    raise RuntimeError("DL model did not produce a valid checkpoint.")

model.load_state_dict(best_state_dict)
model.eval()

print(f"Best validation macro-F1: {best_valid_macro_f1:.4f}")
print(f"DL training time: {training_seconds:.2f} seconds")

Training device: cpu
Epoch 01 | loss=1.3761 | valid_macro_f1=0.7032
Epoch 02 | loss=1.2789 | valid_macro_f1=0.9433
Epoch 03 | loss=0.9952 | valid_macro_f1=0.9312
Epoch 04 | loss=0.5157 | valid_macro_f1=0.9562
Epoch 05 | loss=0.1669 | valid_macro_f1=0.9626
Epoch 06 | loss=0.0489 | valid_macro_f1=0.9626
Epoch 07 | loss=0.0204 | valid_macro_f1=0.9625
Epoch 08 | loss=0.0120 | valid_macro_f1=0.9626
Epoch 09 | loss=0.0071 | valid_macro_f1=0.9626
Early stopping triggered.
Best validation macro-F1: 0.9626
DL training time: 2.47 seconds


## 14. Evaluate the DL model

The DL model is evaluated on the same held-out test set as the classical baseline. This keeps the model comparison fair.

In [ ]:
def predict_dl_labels(texts: list[str]) -> list[str]:
    """Predict labels with the PyTorch model in the notebook environment."""
    features = dl_vectorizer.transform(texts).astype(np.float32).toarray()
    tensor = torch.tensor(features, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()

    return [ID_TO_LABEL[pred] for pred in preds]


dl_test_pred = predict_dl_labels(x_test)

dl_latency = measure_latency_ms(
    predict_dl_labels,
    x_test[: min(100, len(x_test))],
    runs=5,
)

dl_result = evaluate_predictions(
    name="small_dl_tfidf_mlp",
    y_true=y_test,
    y_pred=dl_test_pred,
    latency=dl_latency,
    estimated_cost_usd=0.0,
    notes="Small PyTorch MLP trained offline in Colab and exported to ONNX.",
)

display_result(dl_result)


small_dl_tfidf_mlp
Macro-F1:       0.9627
Weighted-F1:    0.9626
Avg latency ms: 2.92
P95 latency ms: 6.71
Cost USD:       0.0
Per-class F1:
  spam       0.9873
  question   0.9268
  lead       0.9630
  escalate   0.9737
Notes: Small PyTorch MLP trained offline in Colab and exported to ONNX.


## 15. Export the DL model to ONNX

The production model-server should not depend on PyTorch. This cell exports the neural model to ONNX and saves the TF-IDF vectorizer separately.

In [ ]:
onnx_artifact_path = ARTIFACT_DIR / "classifier.onnx"
dl_vectorizer_path = ARTIFACT_DIR / "dl_vectorizer.joblib"

model_cpu = model.to("cpu")
model_cpu.eval()

dummy_input = torch.zeros((1, input_dim), dtype=torch.float32)

torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_artifact_path,
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=17,
)

joblib.dump(dl_vectorizer, dl_vectorizer_path)

print(f"Saved ONNX artifact: {onnx_artifact_path}")
print(f"Saved DL vectorizer: {dl_vectorizer_path}")
print(f"ONNX SHA-256: {sha256_file(onnx_artifact_path)}")
print(f"DL vectorizer SHA-256: {sha256_file(dl_vectorizer_path)}")

/tmp/ipykernel_3471/1136832516.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0527 11:08:05.910000 3471 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `IntentMLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `IntentMLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Saved ONNX artifact: /content/concierge_classifier/artifacts/classifier.onnx
Saved DL vectorizer: /content/concierge_classifier/artifacts/dl_vectorizer.joblib
ONNX SHA-256: a564404e55503a03b577cd8134994b59e7beca828134999b93d44cdef9423709
DL vectorizer SHA-256: 7a2f8ff5168b6e114876c6ad69a994731425b35c12af280fef783dc5bc1b6185


## 16. Verify ONNX inference

This cell checks that the exported ONNX model loads with `onnxruntime` and produces usable predictions. This verifies the lean serving path.

In [ ]:
onnx_session = ort.InferenceSession(
    str(onnx_artifact_path),
    providers=["CPUExecutionProvider"],
)


def softmax(array: np.ndarray) -> np.ndarray:
    """Compute softmax probabilities."""
    shifted = array - np.max(array, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


def predict_onnx_labels(texts: list[str]) -> list[str]:
    """Predict labels using the exported ONNX artifact."""
    features = dl_vectorizer.transform(texts).astype(np.float32).toarray()
    logits = onnx_session.run(None, {"features": features})[0]
    pred_ids = np.argmax(logits, axis=1).tolist()
    return [ID_TO_LABEL[pred_id] for pred_id in pred_ids]


onnx_test_pred = predict_onnx_labels(x_test)
onnx_macro_f1 = f1_score(y_test, onnx_test_pred, labels=LABELS, average="macro")

onnx_latency = measure_latency_ms(
    predict_onnx_labels,
    x_test[: min(100, len(x_test))],
    runs=5,
)

print(f"ONNX test macro-F1: {onnx_macro_f1:.4f}")
print(f"ONNX latency: {onnx_latency}")

sample_texts = [
    "Can I talk to a human?",
    "I want to book a demo for my company.",
    "What are your opening hours?",
    "WIN a free prize now!",
]

for text, label in zip(sample_texts, predict_onnx_labels(sample_texts)):
    print(f"{label:10s} | {text}")

ONNX test macro-F1: 0.9627
ONNX latency: {'avg_ms': 7.128094978015724, 'p95_ms': 15.124277299673842}
question   | Can I talk to a human?
lead       | I want to book a demo for my company.
question   | What are your opening hours?
spam       | WIN a free prize now!


## 17. Run the Groq LLM zero-shot baseline

This cell evaluates a hosted Groq model as the LLM zero-shot baseline. Groq is called through its OpenAI-compatible API, so the notebook can use the `openai` Python client with Groq’s base URL. This baseline is used only for comparison against the classical ML and DL/ONNX models.

In [ ]:
# Groq optional configuration.
#
# In Colab, either:
# 1. Add GROQ_API_KEY to Colab Secrets, or
# 2. Uncomment and set it manually below.
#
# os.environ["GROQ_API_KEY"] = "your_groq_key_here"
#
# Optional:
# os.environ["GROQ_MODEL"] = "llama-3.3-70b-versatile"
# os.environ["GROQ_EVAL_LIMIT"] = "80"

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_EVAL_LIMIT = int(os.getenv("GROQ_EVAL_LIMIT", "80"))


def get_groq_api_key() -> str:
    """Read the Groq API key from Colab Secrets or environment variables."""
    api_key = os.getenv("GROQ_API_KEY", "")

    if api_key:
        return api_key

    try:
        from google.colab import userdata

        secret_value = userdata.get("GROQ_API_KEY")
        if secret_value:
            return secret_value
    except Exception:
        pass

    return ""


def extract_label_from_llm_text(content: str) -> str:
    """Extract one valid label from an LLM response."""
    content = content.strip()

    try:
        parsed = json.loads(content)
        label = str(parsed.get("label", "")).strip().lower()
        if label in LABELS:
            return label
    except json.JSONDecodeError:
        pass

    lowered = content.lower()
    for label in LABELS:
        if label in lowered:
            return label

    return "question"


GROQ_API_KEY = get_groq_api_key()

llm_result: ModelResult | None = None
llm_predictions: list[str] = []

llm_eval_df = test_df.sample(
    n=min(GROQ_EVAL_LIMIT, len(test_df)),
    random_state=SEED,
).reset_index(drop=True)

if not GROQ_API_KEY:
    print("GROQ_API_KEY is not set. Skipping Groq LLM baseline.")
else:
    from openai import OpenAI

    client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
    )

    system_prompt = """
You classify visitor messages for a multi-tenant website concierge.

Allowed labels:
- spam: advertising, scams, giveaways, irrelevant mass messages, or suspicious promotional text
- question: factual or informational questions that can be answered from business content
- lead: buying intent, booking intent, quote requests, consultation requests, or sales follow-up
- escalate: urgent support, human handoff, complaint, fraud, lost/stolen issue, cancellation, or manual review

Return only JSON with this exact schema:
{"label": "spam|question|lead|escalate"}
""".strip()

    start = time.perf_counter()

    for idx, row in llm_eval_df.iterrows():
        user_prompt = f"Message: {row['text']}"

        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
        )

        content = response.choices[0].message.content or ""
        label = extract_label_from_llm_text(content)
        llm_predictions.append(label)

        if (idx + 1) % 10 == 0:
            print(f"Classified {idx + 1}/{len(llm_eval_df)} examples")

    elapsed = time.perf_counter() - start
    avg_latency_ms = (elapsed / len(llm_eval_df)) * 1000

    llm_result = evaluate_predictions(
        name=f"groq_zero_shot_{GROQ_MODEL}",
        y_true=llm_eval_df["label"].tolist(),
        y_pred=llm_predictions,
        latency={
            "avg_ms": avg_latency_ms,
            "p95_ms": avg_latency_ms,
        },
        estimated_cost_usd=None,
        notes=(
            "Groq hosted-API zero-shot baseline. Evaluated on a subset by "
            "default to control cost and latency."
        ),
    )

    display_result(llm_result)

Classified 10/80 examples
Classified 20/80 examples
Classified 30/80 examples
Classified 40/80 examples
Classified 50/80 examples
Classified 60/80 examples
Classified 70/80 examples
Classified 80/80 examples

groq_zero_shot_llama-3.3-70b-versatile
Macro-F1:       0.7830
Weighted-F1:    0.7810
Avg latency ms: 1512.50
P95 latency ms: 1512.50
Cost USD:       None
Per-class F1:
  spam       1.0000
  question   0.7636
  lead       0.5185
  escalate   0.8500
Notes: Groq hosted-API zero-shot baseline. Evaluated on a subset by default to control cost and latency.


## 18. Compare models and choose a shipping candidate

The production choice is based on quality, latency, cost, and serving complexity. The LLM baseline is useful for comparison, but the shipped classifier should normally be a local lean model because the router exists to avoid unnecessary LLM calls.

In [ ]:
results = [classical_result, dl_result]
if llm_result is not None:
    results.append(llm_result)

results_payload = {
    result.name: asdict(result)
    for result in results
}

write_json(REPORT_DIR / "eval_results.json", results_payload)

comparison_df = pd.DataFrame([asdict(result) for result in results])
display(
    comparison_df[
        [
            "name",
            "macro_f1",
            "weighted_f1",
            "avg_latency_ms",
            "p95_latency_ms",
            "estimated_cost_usd",
            "notes",
        ]
    ]
)

if classical_result.macro_f1 >= dl_result.macro_f1 - 0.02:
    shipped_model = "classical"
    shipped_reason = (
        "Classical TF-IDF + Logistic Regression is within 0.02 macro-F1 "
        "of the DL model and is simpler, faster, cheaper, and easier to serve."
    )
    serving_method = "sklearn/joblib"
else:
    shipped_model = "dl_onnx"
    shipped_reason = (
        "The ONNX model has a meaningfully better macro-F1 than the classical "
        "baseline, so the extra serving complexity is justified."
    )
    serving_method = "ONNX/onnxruntime"

print_section("Shipping decision")
print(f"Shipped model: {shipped_model}")
print(f"Serving method: {serving_method}")
print(f"Reason: {shipped_reason}")

,name,macro_f1,weighted_f1,avg_latency_ms,p95_latency_ms,estimated_cost_usd,notes
0,classical_tfidf_logistic_regression,0.943199,0.943261,0.946531,1.259268,0.0,Training completed in 0.43 seconds. Exported w...
1,small_dl_tfidf_mlp,0.962705,0.962635,2.923146,6.706128,0.0,Small PyTorch MLP trained offline in Colab and...
2,groq_zero_shot_llama-3.3-70b-versatile,0.783039,0.780989,1512.495435,1512.495435,NaN,Groq hosted-API zero-shot baseline. Evaluated ...



Shipping decision
Shipped model: classical
Serving method: sklearn/joblib
Reason: Classical TF-IDF + Logistic Regression is within 0.02 macro-F1 of the DL model and is simpler, faster, cheaper, and easier to serve.


## 19. Create metadata and the model-server inference contract

The model-server needs label order, model version, confidence threshold, route policy, dataset hash, and artifact hashes. This metadata allows the service to load the correct artifact and return a stable `/predict` response.

In [ ]:
artifact_hashes = {
    "classifier.joblib": sha256_file(classical_artifact_path),
    "classifier.onnx": sha256_file(onnx_artifact_path),
    "dl_vectorizer.joblib": sha256_file(dl_vectorizer_path),
}

metadata = {
    "project_name": PROJECT_NAME,
    "model_version": MODEL_VERSION,
    "task": "visitor_intent_classification",
    "labels": LABELS,
    "label_to_id": LABEL_TO_ID,
    "id_to_label": ID_TO_LABEL,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "dataset_hash": dataset_hash,
    "dataset_sources": [
        "SMS Spam Collection",
        "CLINC150 / clinc_oos",
    ],
    "shipped_model": shipped_model,
    "serving_method": serving_method,
    "shipped_reason": shipped_reason,
    "artifact_hashes": artifact_hashes,
    "results": results_payload,
}

inference_contract = {
    "endpoint": "POST /predict",
    "request": {
        "text": "string",
    },
    "response": {
        "label": "spam | question | lead | escalate",
        "confidence": "float between 0 and 1",
        "scores": {
            "spam": "float",
            "question": "float",
            "lead": "float",
            "escalate": "float",
        },
        "model_version": MODEL_VERSION,
        "route_hint": "drop | rag_search | capture_lead | escalate | agent_handoff",
    },
    "route_policy": {
        "spam": "drop",
        "question": "rag_search",
        "lead": "capture_lead",
        "escalate": "escalate",
        "low_confidence": "agent_handoff",
        "confidence_threshold": CONFIDENCE_THRESHOLD,
    },
}

metadata_path = ARTIFACT_DIR / "classifier_metadata.json"
contract_path = ARTIFACT_DIR / "inference_contract.json"

write_json(metadata_path, metadata)
write_json(contract_path, inference_contract)

print(f"Saved metadata: {metadata_path}")
print(f"Saved inference contract: {contract_path}")

Saved metadata: /content/concierge_classifier/artifacts/classifier_metadata.json
Saved inference contract: /content/concierge_classifier/artifacts/inference_contract.json


## 20. Generate the model card

The model card records the task, public data sources, dataset hash, three-way model comparison, deployment choice, and artifact SHA-256 hashes. This document should be committed with the model-server artifacts.

In [ ]:
llm_summary = "Not run"
if llm_result is not None:
    llm_summary = (
        f"macro-F1={llm_result.macro_f1:.4f}, "
        f"weighted-F1={llm_result.weighted_f1:.4f}, "
        f"avg latency={llm_result.avg_latency_ms:.2f} ms"
    )

source_counts = (
    balanced_df.groupby(["label", "source_dataset"])
    .size()
    .reset_index(name="count")
    .to_dict(orient="records")
)

model_card = f"""# Concierge Intent Classifier Model Card

## Task

Visitor intent classification for the Concierge router.

The model predicts one of four labels:

- `spam` — drop before storage or agent use
- `question` — route to the RAG workflow
- `lead` — route to lead capture workflow
- `escalate` — route to human handoff or escalation workflow

## Product use

The classifier is used as a cheap router before the expensive tool-calling agent path. Low-confidence predictions should be handed to the agent instead of forcing a deterministic workflow.

## Dataset

Public labeled text-classification sources:

- SMS Spam Collection for `spam`
- CLINC150 / `clinc_oos` public intent dataset for `question`, `lead`, and `escalate`

The CLINC150 intent labels are collapsed into Concierge router labels using a transparent mapping in the notebook.

Dataset SHA-256: `{dataset_hash}`

Balanced examples per class: `{target_per_class}`

Source counts:

{json.dumps(source_counts, indent=2)}

## Evaluation

Main metric: macro-F1 on the held-out test set.

| Model | Macro-F1 | Weighted-F1 | Avg latency ms | P95 latency ms | Cost |
|---|---:|---:|---:|---:|---:|
| Classical TF-IDF + Logistic Regression | {classical_result.macro_f1:.4f} | {classical_result.weighted_f1:.4f} | {classical_result.avg_latency_ms:.2f} | {classical_result.p95_latency_ms:.2f} | 0 |
| Small DL MLP exported to ONNX | {dl_result.macro_f1:.4f} | {dl_result.weighted_f1:.4f} | {dl_result.avg_latency_ms:.2f} | {dl_result.p95_latency_ms:.2f} | 0 |
| Hosted-API LLM zero-shot | {llm_summary} | | | | provider-priced |

## Per-class F1

Classical:

{json.dumps(classical_result.per_class_f1, indent=2)}

DL / ONNX:

{json.dumps(dl_result.per_class_f1, indent=2)}

## Deployment choice

- Shipped model: `{shipped_model}`
- Serving method: `{serving_method}`
- Reason: {shipped_reason}

## Artifact hashes

{json.dumps(artifact_hashes, indent=2)}

## Serving notes

The production model-server should load artifacts once at startup and refuse to boot if the artifact SHA-256 does not match this model card. The serving container must not include PyTorch, TensorFlow, or transformers.
"""

model_card_path = ARTIFACT_DIR / "model_card.md"
model_card_path.write_text(model_card, encoding="utf-8")

print(model_card)
print(f"\nSaved model card: {model_card_path}")

# Concierge Intent Classifier Model Card

## Task

Visitor intent classification for the Concierge router.

The model predicts one of four labels:

- `spam` — drop before storage or agent use
- `question` — route to the RAG workflow
- `lead` — route to lead capture workflow
- `escalate` — route to human handoff or escalation workflow

## Product use

The classifier is used as a cheap router before the expensive tool-calling agent path. Low-confidence predictions should be handed to the agent instead of forcing a deterministic workflow.

## Dataset

Public labeled text-classification sources:

- SMS Spam Collection for `spam`
- CLINC150 / `clinc_oos` public intent dataset for `question`, `lead`, and `escalate`

The CLINC150 intent labels are collapsed into Concierge router labels using a transparent mapping in the notebook.

Dataset SHA-256: `894e9a9c6ac512789458f5385839347eb68ace52fc2b6158e89ceb53d54cf350`

Balanced examples per class: `264`

Source counts:

[
  {
    "label": "escalat

## 21. Generate starter inference code for the model-server

This cell creates a reference inference module for the lean model-server. It loads the chosen artifact once and returns the `/predict` response expected by the backend.

In [ ]:
starter_inference_code = r'''
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import onnxruntime as ort


ARTIFACT_DIR = Path(__file__).resolve().parents[1] / "artifacts"
METADATA_PATH = ARTIFACT_DIR / "classifier_metadata.json"


def _softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


class IntentClassifier:
    """Lean inference wrapper for the Concierge intent classifier."""

    def __init__(self) -> None:
        self.metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
        self.labels = self.metadata["labels"]
        self.model_version = self.metadata["model_version"]
        self.confidence_threshold = self.metadata["confidence_threshold"]
        self.shipped_model = self.metadata["shipped_model"]

        if self.shipped_model == "classical":
            self.model = joblib.load(ARTIFACT_DIR / "classifier.joblib")
            self.vectorizer = None
            self.onnx_session = None
        elif self.shipped_model == "dl_onnx":
            self.model = None
            self.vectorizer = joblib.load(ARTIFACT_DIR / "dl_vectorizer.joblib")
            self.onnx_session = ort.InferenceSession(
                str(ARTIFACT_DIR / "classifier.onnx"),
                providers=["CPUExecutionProvider"],
            )
        else:
            raise ValueError(f"Unsupported shipped model: {self.shipped_model}")

    def predict(self, text: str) -> dict[str, Any]:
        """Predict one visitor message."""
        if self.shipped_model == "classical":
            probabilities = self.model.predict_proba([text])[0]
            class_order = self.model.classes_.tolist()
            scores = {
                label: float(probabilities[class_order.index(label)])
                if label in class_order
                else 0.0
                for label in self.labels
            }
        else:
            features = self.vectorizer.transform([text]).astype(np.float32).toarray()
            logits = self.onnx_session.run(None, {"features": features})[0]
            probabilities = _softmax(logits)[0]
            scores = {
                label: float(probabilities[idx])
                for idx, label in enumerate(self.labels)
            }

        label = max(scores, key=scores.get)
        confidence = scores[label]

        route_hint_by_label = {
            "spam": "drop",
            "question": "rag_search",
            "lead": "capture_lead",
            "escalate": "escalate",
        }

        route_hint = route_hint_by_label[label]
        if confidence < self.confidence_threshold:
            route_hint = "agent_handoff"

        return {
            "label": label,
            "confidence": confidence,
            "scores": scores,
            "model_version": self.model_version,
            "route_hint": route_hint,
        }
'''

starter_path = EXPORT_DIR / "modelserver_inference_reference.py"
starter_path.write_text(starter_inference_code.strip() + "\n", encoding="utf-8")

print(f"Saved starter inference reference: {starter_path}")

Saved starter inference reference: /content/concierge_classifier/export/modelserver_inference_reference.py


## 22. Package artifacts for download

This cell packages the trained models, ONNX export, vectorizer, metadata, model card, evaluation results, dataset splits, threshold file, and starter inference module into one zip file.

In [ ]:
package_dir = EXPORT_DIR / "concierge_classifier_package"
if package_dir.exists():
    shutil.rmtree(package_dir)

(package_dir / "model-server" / "artifacts").mkdir(parents=True, exist_ok=True)
(package_dir / "model-server" / "app").mkdir(parents=True, exist_ok=True)
(package_dir / "ci").mkdir(parents=True, exist_ok=True)
(package_dir / "data" / "processed").mkdir(parents=True, exist_ok=True)
(package_dir / "docs").mkdir(parents=True, exist_ok=True)

for path in [
    classical_artifact_path,
    onnx_artifact_path,
    dl_vectorizer_path,
    metadata_path,
    contract_path,
]:
    shutil.copy2(path, package_dir / "model-server" / "artifacts" / path.name)

shutil.copy2(model_card_path, package_dir / "model-server" / "app" / "model_card.md")
shutil.copy2(model_card_path, package_dir / "docs" / "MODEL_CARD.md")

shutil.copy2(
    REPORT_DIR / "eval_results.json",
    package_dir / "ci" / "classifier_eval_results.json",
)

for path in [train_path, valid_path, test_path, full_path]:
    shutil.copy2(path, package_dir / "data" / "processed" / path.name)

shutil.copy2(starter_path, package_dir / "model-server" / "app" / "inference.py")

thresholds = {
    "classifier": {
        "macro_f1_min": round(
            max(0.50, min(classical_result.macro_f1, dl_result.macro_f1) - 0.05),
            4,
        ),
        "p95_latency_ms_max": 100.0,
    },
    "redteam": {
        "required_refusal_rate": 1.0,
    },
}

threshold_path = package_dir / "ci" / "eval_thresholds.yaml"
threshold_path.write_text(yaml.safe_dump(thresholds, sort_keys=False), encoding="utf-8")

zip_path = EXPORT_DIR / "concierge_classifier_artifacts.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in package_dir.rglob("*"):
        if file_path.is_file():
            zip_file.write(file_path, file_path.relative_to(package_dir))

print(f"Created package: {zip_path}")
print(f"Package size MB: {zip_path.stat().st_size / (1024 * 1024):.2f}")

print("\nPackage contents:")
for file_path in sorted(package_dir.rglob("*")):
    if file_path.is_file():
        print(file_path.relative_to(package_dir))

Created package: /content/concierge_classifier/export/concierge_classifier_artifacts.zip
Package size MB: 0.27

Package contents:
ci/classifier_eval_results.json
ci/eval_thresholds.yaml
data/processed/classifier_full.csv
data/processed/classifier_test.csv
data/processed/classifier_train.csv
data/processed/classifier_valid.csv
docs/MODEL_CARD.md
model-server/app/inference.py
model-server/app/model_card.md
model-server/artifacts/classifier.joblib
model-server/artifacts/classifier.onnx
model-server/artifacts/classifier_metadata.json
model-server/artifacts/dl_vectorizer.joblib
model-server/artifacts/inference_contract.json


## 23. Download the exported package

This cell downloads the zip package from Colab. The package contains the trained classifier artifacts, ONNX export, model card, evaluation results, processed dataset splits, threshold YAML, and a starter inference module.

In [ ]:
try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print(f"Not running in Colab. Download manually from: {zip_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 24. Final router sanity test

This final cell tests the chosen shipped model on messages that represent each router path. The expected route hints are `drop`, `rag_search`, `capture_lead`, `escalate`, or `agent_handoff` for low-confidence cases.

In [ ]:
def classical_predict_with_route(text: str) -> dict[str, Any]:
    """Predict one message and return router output using the classical model."""
    probabilities = classical_pipeline.predict_proba([text])[0]
    class_order = classical_pipeline.classes_.tolist()

    scores = {
        label: float(probabilities[class_order.index(label)])
        if label in class_order
        else 0.0
        for label in LABELS
    }

    label = max(scores, key=scores.get)
    confidence = scores[label]

    route_hint_by_label = {
        "spam": "drop",
        "question": "rag_search",
        "lead": "capture_lead",
        "escalate": "escalate",
    }

    route_hint = route_hint_by_label[label]
    if confidence < CONFIDENCE_THRESHOLD:
        route_hint = "agent_handoff"

    return {
        "text": text,
        "label": label,
        "confidence": round(confidence, 4),
        "route_hint": route_hint,
        "scores": {key: round(value, 4) for key, value in scores.items()},
    }


def onnx_predict_with_route(text: str) -> dict[str, Any]:
    """Predict one message and return router output using the ONNX model."""
    features = dl_vectorizer.transform([text]).astype(np.float32).toarray()
    logits = onnx_session.run(None, {"features": features})[0]
    probabilities = softmax(logits)[0]

    scores = {
        label: float(probabilities[idx])
        for idx, label in enumerate(LABELS)
    }

    label = max(scores, key=scores.get)
    confidence = scores[label]

    route_hint_by_label = {
        "spam": "drop",
        "question": "rag_search",
        "lead": "capture_lead",
        "escalate": "escalate",
    }

    route_hint = route_hint_by_label[label]
    if confidence < CONFIDENCE_THRESHOLD:
        route_hint = "agent_handoff"

    return {
        "text": text,
        "label": label,
        "confidence": round(confidence, 4),
        "route_hint": route_hint,
        "scores": {key: round(value, 4) for key, value in scores.items()},
    }


sanity_messages = [
    "WIN a free coupon now, click this link immediately!",
    "What are your opening hours this weekend?",
    "My company wants to book a demo and discuss pricing.",
    "Please connect me to a human support agent.",
    "I need help with something complicated and maybe pricing too.",
]

predict_with_route = (
    classical_predict_with_route
    if shipped_model == "classical"
    else onnx_predict_with_route
)

sanity_outputs = [predict_with_route(message) for message in sanity_messages]
display(pd.DataFrame(sanity_outputs))

,text,label,confidence,route_hint,scores
0,"WIN a free coupon now, click this link immedia...",spam,0.7314,drop,"{'spam': 0.7314, 'question': 0.1114, 'lead': 0..."
1,What are your opening hours this weekend?,question,0.4584,agent_handoff,"{'spam': 0.338, 'question': 0.4584, 'lead': 0...."
2,My company wants to book a demo and discuss pr...,lead,0.3225,agent_handoff,"{'spam': 0.227, 'question': 0.2167, 'lead': 0...."
3,Please connect me to a human support agent.,lead,0.3451,agent_handoff,"{'spam': 0.2363, 'question': 0.2458, 'lead': 0..."
4,I need help with something complicated and may...,spam,0.3093,agent_handoff,"{'spam': 0.3093, 'question': 0.231, 'lead': 0...."
